# Purpose of Script

The purpose of this script is to simply get familiar with Google Cloud, the formatting, and general data models.

Resource https://omophub.com/blog/omop-data-model$0

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("ggplot")

# from google.colab.sql import bigquery as bq

from google.cloud import bigquery
from IPython.display import Image, Markdown

CLIENT = bigquery.Client(project="project-d60bc1f8-9487-4a43-958")

Influenza SNOMED CT Codes found through the url above

In [ ]:
## Stroke: https://evsexplore.semantics.cancer.gov/evsexplore/concept/snomedct_us/230690007$0
## Cerebrovascular accident ( Code - 230690007 )

## Heart attack: https://evsexplore.semantics.cancer.gov/evsexplore/concept/snomedct_us/22298006$0
## Myocardial infarction ( Code - 22298006 )

concept_codes = {'cerebro_accident': '230690007',
                  'myocardial_infarction': '22298006'}


concept_ids = {'cerebro_accident': 381316,
                'myocardial_infarction': 4329847}

## concept_code -> concept_id (lookup through concept table by searching concept_code)
## manually pasted above just for easy / explicit viewing

### For the tables viewed below, the concept 'code' as defined by the government website is used to query the `concept` table, and then afterwards, not used again.

In [ ]:
### beter direct in python
query = f"""
            SELECT *
            FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept`
            WHERE 1=1
            AND `vocabulary_id` = 'SNOMED'
            AND `concept_code` in {tuple(list(concept_codes.values()))}
            LIMIT 10
            ;"""

df = CLIENT.query(query).to_dataframe()
display(df)

In [152]:
query = f"""
            
           
            SELECT *
            FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_era`
            WHERE 1=1
            AND `person_id` = 1137719
            AND `condition_concept_id` IN (
                312327,
                438447,
                434376,
                438170,
                444406, 
                439693, 
                436706
                )
            ORDER BY `condition_era_start_date`
            LIMIT 100
            ;"""

df = CLIENT.query(query).to_dataframe()
display(df)

,condition_era_id,person_id,condition_concept_id,condition_era_start_date,condition_era_end_date,condition_occurrence_count
0,909558197,1137719,312327,2008-01-15,2008-01-15,1
1,909558299,1137719,434376,2008-01-15,2008-01-15,1
2,909558323,1137719,438170,2008-03-01,2008-03-04,1
3,909558326,1137719,438447,2008-04-02,2008-04-23,2
4,909558359,1137719,444406,2008-04-02,2008-06-19,22
5,909558198,1137719,312327,2008-04-25,2008-06-17,12
6,909558300,1137719,434376,2008-06-17,2008-07-01,3
7,909558334,1137719,439693,2008-06-19,2008-06-19,1
8,909558327,1137719,438447,2008-08-17,2008-08-17,1
9,909558199,1137719,312327,2008-08-17,2008-10-21,7


In [153]:
test = df.drop_duplicates(subset=['person_id', 'condition_era_start_date']).reset_index(drop=True).reset_index()
test = test.rename(columns={'index': 'flu_case_number'})
test

,flu_case_number,condition_era_id,person_id,condition_concept_id,condition_era_start_date,condition_era_end_date,condition_occurrence_count
0,0,909558197,1137719,312327,2008-01-15,2008-01-15,1
1,1,909558323,1137719,438170,2008-03-01,2008-03-04,1
2,2,909558326,1137719,438447,2008-04-02,2008-04-23,2
3,3,909558198,1137719,312327,2008-04-25,2008-06-17,12
4,4,909558300,1137719,434376,2008-06-17,2008-07-01,3
5,5,909558334,1137719,439693,2008-06-19,2008-06-19,1
6,6,909558327,1137719,438447,2008-08-17,2008-08-17,1
7,7,909558324,1137719,438170,2008-09-24,2008-09-24,1
8,8,909558301,1137719,434376,2008-10-21,2008-10-21,1
9,9,909558328,1137719,438447,2008-11-21,2008-11-21,1


In [160]:
import plotly.express as px
import datetime

start = pd.to_datetime(test['condition_era_start_date'])
end = pd.to_datetime(test['condition_era_end_date'].apply(lambda x: x + datetime.timedelta(days=1)))

test['start'] = start
test['end'] = end

px.timeline(test,  y = 'flu_case_number', x_start='start', x_end = 'end',
                            title="Automatic Labels Based on Data Frame Column")
# px.xlabel("Date Sick")


# start

## Condition Occurrence Preview

CONDITION_OCCURRENCE: This is where you’ll find all patient diagnoses, from chronic illnesses to acute infections. Each row represents a specific condition like 'Type 2 diabetes' or 'acute bronchitis,' tied to a patient and, usually, to the visit when it was first diagnosed.

In [ ]:
# query = """
#           SELECT *
#         FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
#         limit 5
#         """

# df = CLIENT.query(query).to_dataframe()
# display(df)

query = f"""
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE 1=1
        AND
        `condition_concept_id` in {tuple(concept_ids.values())} -- may be too low on the tree
        """

df = CLIENT.query(query).to_dataframe()
display(df)

# query = f"""
#         SELECT *
#         FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
#         WHERE 1=1
#         AND `condition_concept_id` = 230690007
#         limit 5
#         """

# df = CLIENT.query(query).to_dataframe()
# display(df)

## Domain Preview

## Concept Relationships

In [ ]:
print(f"From government website (concept 'codes'):")
print(concept_codes)
print("\nFrom OMOP concept table (concept 'ids'):")
print(concept_ids)

In [ ]:
query = f"""
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_relationship`
        WHERE 1 = 1
        AND `concept_id_1` IN {tuple(concept_ids.values())}
        ;"""

df = CLIENT.query(query).to_dataframe()

## all "Condition" in domain_id - maybe always - maybe inconsistent
display(df)

## Concept Ancestors

In [ ]:
query = f"""
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ;"""

df = CLIENT.query(query).to_dataframe()

## all "Condition" in domain_id - maybe always - maybe inconsistent
display(df)

In [ ]:
query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        disease_happened as (
        SELECT *, EXTRACT(MONTH from `condition_start_date`) AS condition_start_date_month
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `condition_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        )

        select *
        FROM disease_happened
        """

df = CLIENT.query(query).to_dataframe()
display(df)

In [ ]:
month_order = df['condition_start_date_month'].value_counts().sort_index().reset_index()

fig, ax = plt.subplots(figsize=(10,4))

plt.bar(month_order['condition_start_date_month'], month_order['count'])
plt.xticks(np.arange(1, 13, 1))
plt.xlabel("Month")
plt.ylabel("Number of Stroke/Heart Attacks")
plt.title("Diagnoses per Month")

plt.show()

In [ ]:
## unique people and location

query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        diagnosis_happened as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `condition_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        ),

        people_w_diagnosis as (

        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.person`
        WHERE `person_id` IN
                    (
                    SELECT DISTINCT(f.`person_id`)
                    FROM diagnosis_happened f
                    )

        ),

        people_w_location as (
        SELECT *
        FROM people_w_diagnosis pf
        JOIN `bigquery-public-data.cms_synthetic_patient_data_omop.location` l
        using(`location_id`)
        )

        select *
        FROM people_w_location
        """

df = CLIENT.query(query).to_dataframe()
display(df)

In [ ]:
display(df['race_concept_id'].value_counts())

display(df['ethnicity_concept_id'].value_counts())

display(df['state'].value_counts().sort_values(ascending=False))

In [ ]:
query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        subtype_concept_details as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept`
        WHERE `concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        )

        select *
        FROM subtype_concept_details
        """

df = CLIENT.query(query).to_dataframe()
display(df)

In [ ]:
query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        diagnosis_happened as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `condition_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        )

        select COUNT(DISTINCT(`person_id`)) as `people_w_diagnosis`
        FROM diagnosis_happened
        """

df = CLIENT.query(query).to_dataframe()
display(df)

In [ ]:
query = f"""

        WITH cardiac_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        diagnosis_happened as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `condition_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM cardiac_subtypes f
                    )
        )

        select `person_id`, COUNT(`person_id`) as `num_times_w_cardiac_problem`
        FROM diagnosis_happened
        GROUP BY `person_id`
        ORDER BY num_times_w_cardiac_problem DESC
        """

df = CLIENT.query(query).to_dataframe()
display(df)

In [ ]:
df['num_times_w_cardiac_problem'].value_counts().sort_index(ascending=False)

In [ ]:
query = """
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `person_id` = 1137719
        ORDER BY `condition_start_date`
        """

df = CLIENT.query(query).to_dataframe()
display(df)


query = """
        WITH cardiac_often_person AS
        (SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.condition_occurrence`
        WHERE `person_id` = 1137719
        ),

        date_ranges as (
        SELECT min(condition_start_date) as `first_time_cardiac_problem`,
              max(condition_end_date) as `last_time_cardiac_problem`
              FROM cardiac_often_person f

        )

        SELECT *
        FROM date_ranges
        """

df = CLIENT.query(query).to_dataframe()
display(df)

In [ ]:
query  = """SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.death`
        LIMIT 100"""

df = CLIENT.query(query).to_dataframe()

display(df)


query  = """SELECT COUNT(DISTINCT(`person_id`)) as `num_deaths`
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.death`
        LIMIT 100"""

df = CLIENT.query(query).to_dataframe()

display(df)

In [ ]:
query = f"""

        WITH flu_subtypes AS
        ( SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.concept_ancestor`
        WHERE 1 = 1
        AND `ancestor_concept_id` IN {tuple(concept_ids.values())}
        ),

        death_from_flu as (
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.death`
        WHERE `cause_concept_id` IN
                    (
                    SELECT DISTINCT(f.`descendant_concept_id`)
                    FROM flu_subtypes f
                    )
        OR `cause_concept_id` IN {tuple(concept_ids.values())}
        )

        select *
        FROM death_from_flu
        """

df = CLIENT.query(query).to_dataframe()
display(df)

In [ ]:
query = """
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.person`
        WHERE `person_id` = 1137719
        """

df = CLIENT.query(query).to_dataframe()
display(df) ## year of birth 1919 ~ 107 years old

In [ ]:
query = """
        SELECT *
        FROM `bigquery-public-data.cms_synthetic_patient_data_omop.location`
        WHERE `location_id` = 1663
        """
df = CLIENT.query(query).to_dataframe()
display(df) # Missouri

## Miscellanous

In [ ]:
# bq.run("""SELECT
#     m.person_id,
#     m.measurement_date,
#     m.value_as_number
# FROM
#     `bigquery-public-data.cms_synthetic_patient_data_omop.measurement` m
# JOIN
#     `bigquery-public-data.cms_synthetic_patient_data_omop.concept` c

#     ON m.measurement_concept_id = c.concept_id
# WHERE
#     c.concept_name = 'Systolic blood pressure'
#     AND m.value_as_number > 140
#     AND m.measurement_date >= '2023-01-01';""")


# ## measurement table doesn't exist in the US?